# ⚙️ Vanilla RNN Cell 밑바닥부터 구현하기

In [1]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np

# 시드 고정
torch.manual_seed(42)

## 1. PyTorch 텐서 곱셈으로 RNN 셀 수식 직접 짜보기
`nn.RNN`을 사용하지 않고, 수학 공식 $h_t = \tanh(x_t W_{xh}^T + h_{t-1} W_{hh}^T + b)$ 를 그대로 텐서 코드로 구현합니다.

In [2]:
class VanillaRNNCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(VanillaRNNCell, self).__init__()
        self.hidden_size = hidden_size
        
        # 가중치 행렬 초기화 (정규 분포)
        self.W_xh = nn.Parameter(torch.randn(hidden_size, input_size) * 0.1)
        self.W_hh = nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.1)
        self.bias = nn.Parameter(torch.zeros(hidden_size))
        
    def forward(self, x_t, h_prev):
        # x_t: (batch_size, input_size)
        # h_prev: (batch_size, hidden_size)
        
        # 선형 변환 및 행렬 곱
        # (batch_size, input_size) @ (input_size, hidden_size) = (batch_size, hidden_size)
        xw = torch.matmul(x_t, self.W_xh.t())
        hh = torch.matmul(h_prev, self.W_hh.t())
        
        # Tanh 활성화 함수를 통과하여 현재 시점의 은닉 상태 계산
        h_t = torch.tanh(xw + hh + self.bias)
        return h_t

# 테스트
batch_size, seq_len, input_size, hidden_size = 32, 10, 50, 128
rnn_cell = VanillaRNNCell(input_size, hidden_size)

x_t = torch.randn(batch_size, input_size) # 현재 시점의 단어 임베딩
h_prev = torch.zeros(batch_size, hidden_size) # 이전 시점의 은닉 상태 (초기엔 0)

h_t = rnn_cell(x_t, h_prev)
print(f"계산된 Hidden State 차원: {h_t.shape} (기대값: 32, 128)")

계산된 Hidden State 차원: torch.Size([32, 128]) (기대값: 32, 128)


## 2. 시간 순서대로 시퀀스 처리하기 (루프)
위에서 만든 셀을 이용해 전체 문장(seq_len=10)을 순차적으로 처리하는 루프를 만듭니다.

In [3]:
x_seq = torch.randn(batch_size, seq_len, input_size)
hidden_states = []
h_t = torch.zeros(batch_size, hidden_size) # 초기화

for t in range(seq_len):
    # 각 시점(t)마다 단어를 하나씩 뽑아서 RNN 셀에 넣음
    x_t = x_seq[:, t, :]
    h_t = rnn_cell(x_t, h_t)
    hidden_states.append(h_t)

# 모든 시점의 은닉 상태를 쌓음
hidden_states = torch.stack(hidden_states, dim=1)
print(f"전체 문장 처리 후 Hidden States 차원: {hidden_states.shape} (기대값: 32, 10, 128)")

전체 문장 처리 후 Hidden States 차원: torch.Size([32, 10, 128]) (기대값: 32, 10, 128)


## 3. Teacher Forcing (교사 강요) 확률적 시뮬레이션
디코더가 학습할 때, 이전 예측값이 틀리더라도 정답(Target)을 억지로 넣어주어 학습 궤도를 이탈하지 않게 만드는 기법입니다.

In [4]:
import random

def decoder_step_simulation(targets, teacher_forcing_ratio=0.5):
    seq_len = len(targets)
    predictions = []
    
    # 첫 입력은 <SOS> 토큰이라 가정 (여기서는 0으로 시뮬레이션)
    current_input = 0 
    
    for t in range(seq_len):
        # 디코더 모델이 예측을 했다고 가정 (여기서는 단순 랜덤값 + 현재 입력값)
        prediction = current_input + random.randint(-1, 1)
        predictions.append(prediction)
        
        # 교사 강요 확률에 따라 다음 시점의 입력을 결정!
        use_teacher_forcing = random.random() < teacher_forcing_ratio
        
        if use_teacher_forcing:
            current_input = targets[t] # 💡 정답을 억지로 떠먹여 줌!
            print(f"[시점 {t}] 예측값: {prediction} | 다음 입력: 정답({targets[t]}) 사용 (Teacher Forcing!)")
        else:
            current_input = prediction # 💡 자기가 방금 예측한 값을 다음 입력으로 씀
            print(f"[시점 {t}] 예측값: {prediction} | 다음 입력: 예측값({prediction}) 사용")
            
targets = [10, 20, 30, 40, 50]
print(f"실제 정답 타겟: {targets}\n")
decoder_step_simulation(targets, teacher_forcing_ratio=0.5)

실제 정답 타겟: [10, 20, 30, 40, 50]

[시점 0] 예측값: -1 | 다음 입력: 정답(10) 사용 (Teacher Forcing!)
[시점 1] 예측값: 10 | 다음 입력: 예측값(10) 사용
[시점 2] 예측값: 9 | 다음 입력: 예측값(9) 사용
[시점 3] 예측값: 8 | 다음 입력: 정답(40) 사용 (Teacher Forcing!)
[시점 4] 예측값: 39 | 다음 입력: 예측값(39) 사용
